# Statistics for ML Quality: Bootstrap Confidence Intervals & Drift Detection

**Dataset:** [Wisconsin Breast Cancer diagnostic dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) (`sklearn.datasets.load_breast_cancer`) — 569 real patient tumor samples, 30 numeric features derived from digitized cell-nuclei images, labeled malignant/benign.

## Problem statement

A single number like "accuracy = 0.94" is not an engineering spec — it's a point estimate with no sense of how much it would wobble if you drew a different test set, and no information about whether tomorrow's incoming data even resembles the data that number was measured on. As an ML quality engineer, your job is to put error bars on that number and to catch, statistically, when the input distribution has shifted under the model's feet.

In this exercise you will:

1. Train a baseline classifier (`LogisticRegression`) on a train/test split of the breast cancer dataset.
2. Write a **bootstrap confidence interval** function and use it to quantify the uncertainty around accuracy, F1, and ROC-AUC on the test set — not just the point estimate.
3. Split the test set into two "production batches" using a **real, naturally-occurring feature cut** (not a synthetic injection) that produces one batch statistically similar to training data and one that differs.
4. Run a **two-sample Kolmogorov–Smirnov test** per feature between the training distribution and each batch to flag which features show statistically significant drift (with a multiple-testing correction), and report which batch is flagged as drifted.
5. Check whether the batch flagged as "drifted" also shows a wider or lower-centered bootstrap CI for the evaluation metrics — i.e., connect the statistical drift signal to an actual quality risk.

**Concretely, you should produce:**
- A `bootstrap_ci(values_fn, n_boot=2000, ci=0.95)`-style function returning `(point_estimate, lower, upper)` for a given metric.
- A table of (metric, point estimate, 95% CI) for accuracy, F1, and ROC-AUC on the untouched test set.
- A per-feature KS-test table (statistic, p-value, Bonferroni-adjusted significance) comparing train vs. each of the two production batches.
- A short written verdict (2-4 sentences, in a markdown cell) on which batch you'd page someone about, and why, using both the drift test results and the metric CIs as evidence.


## Setup

Run this cell as-is — it loads the data and does the train/test split.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy import stats

RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
X = data.frame[data.feature_names]
y = data.frame["target"]  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(X_train.shape, X_test.shape)

## Step 1 — Baseline model

TODO: fit a `LogisticRegression` on `X_train_s` / `y_train`, predict on `X_test_s`, and get both
hard predictions (`y_pred`) and predicted probabilities for the positive class (`y_proba`), which
you'll need for ROC-AUC.

In [ ]:
# TODO: train the model
# clf = LogisticRegression(...)
# clf.fit(...)

# TODO: predictions on the test set
# y_pred = ...
# y_proba = ...   # probability of class 1, needed for roc_auc_score

# point_acc = accuracy_score(y_test, y_pred)
# point_f1 = f1_score(y_test, y_pred)
# point_auc = roc_auc_score(y_test, y_proba)
# print(point_acc, point_f1, point_auc)


## Step 2 — Bootstrap confidence intervals

TODO: implement `bootstrap_ci`. It should:
- Resample **paired** `(y_true[i], y_pred_or_proba[i])` indices with replacement, `n_boot` times.
- Apply `metric_fn(y_true_sample, pred_sample)` on each resample.
- Return `(point_estimate, lower, upper)` using the percentile method for the given `ci` level
  (e.g. for `ci=0.95`, the 2.5th and 97.5th percentiles of the bootstrap distribution).

Guard against a degenerate resample (e.g. a bootstrap sample with only one class present, which
breaks `roc_auc_score`) by skipping that resample rather than crashing.

In [ ]:
def bootstrap_ci(y_true, preds, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    \"\"\"TODO: implement.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
    preds : array-like of shape (n_samples,) - hard labels or probabilities, whatever metric_fn expects
    metric_fn : callable(y_true_sample, preds_sample) -> float
    n_boot : int
    ci : float, confidence level, e.g. 0.95

    Returns
    -------
    (point_estimate, lower, upper)
    \"\"\"
    # rng = np.random.default_rng(random_state)
    # y_true = np.asarray(y_true)
    # preds = np.asarray(preds)
    # n = len(y_true)
    # point = metric_fn(y_true, preds)
    # boot_scores = []
    # for _ in range(n_boot):
    #     idx = rng.integers(0, n, n)
    #     yt, pr = y_true[idx], preds[idx]
    #     if len(np.unique(yt)) < 2:
    #         continue
    #     boot_scores.append(metric_fn(yt, pr))
    # alpha = (1 - ci) / 2
    # lower, upper = np.percentile(boot_scores, [100 * alpha, 100 * (1 - alpha)])
    # return point, lower, upper
    raise NotImplementedError


# TODO: compute CIs for accuracy, F1 (both use y_pred), and ROC-AUC (uses y_proba)
# acc_ci = bootstrap_ci(y_test.values, y_pred, accuracy_score)
# f1_ci = bootstrap_ci(y_test.values, y_pred, f1_score)
# auc_ci = bootstrap_ci(y_test.values, y_proba, roc_auc_score)
# pd.DataFrame([
#     {"metric": "accuracy", "point": acc_ci[0], "lower": acc_ci[1], "upper": acc_ci[2]},
#     {"metric": "f1", "point": f1_ci[0], "lower": f1_ci[1], "upper": f1_ci[2]},
#     {"metric": "roc_auc", "point": auc_ci[0], "lower": auc_ci[1], "upper": auc_ci[2]},
# ])


## Step 3 — Build two "production batches" from a real feature split

Instead of injecting synthetic drift, split `X_test` using the **median of `"worst area"`** (a real
feature already in the dataset) computed on `X_train`:

- `batch_low`: test rows where `worst area <= train_median`
- `batch_high`: test rows where `worst area > train_median`

Because tumor size/texture features are correlated with malignancy, one of these batches should end
up looking a lot more like the training distribution than the other — that's the "drift" you'll
detect statistically, using a real subpopulation rather than a synthetic shift.

TODO: compute `train_median = X_train["worst area"].median()` and build `batch_low`, `batch_high`
(as index masks into `X_test` / `y_test` / `X_test_s`).

In [ ]:
# TODO
# train_median = X_train["worst area"].median()
# low_mask = X_test["worst area"] <= train_median
# high_mask = ~low_mask
# print(low_mask.sum(), high_mask.sum())


## Step 4 — Per-feature KS test: train vs. each batch

TODO: for every column in `X.columns`, run `scipy.stats.ks_2samp(X_train[col], X_test[col][mask])`
for each batch, collect `(feature, batch, statistic, p_value)` into a DataFrame, then flag
significance using a **Bonferroni-corrected** threshold (`0.05 / n_features`) instead of the raw
0.05, since you're running one test per feature.

In [ ]:
# TODO
# n_features = len(X.columns)
# alpha_corrected = 0.05 / n_features
# rows = []
# for col in X.columns:
#     for batch_name, mask in [("low", low_mask), ("high", high_mask)]:
#         stat, p = stats.ks_2samp(X_train[col], X_test.loc[mask, col])
#         rows.append({"feature": col, "batch": batch_name, "statistic": stat, "p_value": p,
#                      "significant_bonferroni": p < alpha_corrected})
# ks_results = pd.DataFrame(rows)
# ks_results.groupby("batch")["significant_bonferroni"].sum()


## Step 5 — Metric CIs per batch + verdict

TODO: recompute the Step 2 bootstrap CIs (accuracy, F1, ROC-AUC) separately for `batch_low` and
`batch_high` (reuse `bootstrap_ci`, just slice `y_test`/`y_pred`/`y_proba` by `low_mask`/`high_mask`
first). Then write your verdict.

In [ ]:
# TODO: per-batch metric CIs


**Your verdict (2-4 sentences):** Which batch would you flag for a drift alert, and does the
drop (if any) in bootstrap CI for accuracy/F1/AUC support that alert, or does the model hold up fine
despite the statistical drift? Write your answer here.

> *(replace this with your own answer)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Step 1 — Baseline model**

```python
clf = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
y_proba = clf.predict_proba(X_test_s)[:, 1]

point_acc = accuracy_score(y_test, y_pred)
point_f1 = f1_score(y_test, y_pred)
point_auc = roc_auc_score(y_test, y_proba)
print(point_acc, point_f1, point_auc)
```

**Step 2 — Bootstrap CI**

```python
def bootstrap_ci(y_true, preds, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    preds = np.asarray(preds)
    n = len(y_true)
    point = metric_fn(y_true, preds)
    boot_scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, pr = y_true[idx], preds[idx]
        if len(np.unique(yt)) < 2:
            continue
        boot_scores.append(metric_fn(yt, pr))
    alpha = (1 - ci) / 2
    lower, upper = np.percentile(boot_scores, [100 * alpha, 100 * (1 - alpha)])
    return point, lower, upper

acc_ci = bootstrap_ci(y_test.values, y_pred, accuracy_score)
f1_ci = bootstrap_ci(y_test.values, y_pred, f1_score)
auc_ci = bootstrap_ci(y_test.values, y_proba, roc_auc_score)

pd.DataFrame([
    {"metric": "accuracy", "point": acc_ci[0], "lower": acc_ci[1], "upper": acc_ci[2]},
    {"metric": "f1", "point": f1_ci[0], "lower": f1_ci[1], "upper": f1_ci[2]},
    {"metric": "roc_auc", "point": auc_ci[0], "lower": auc_ci[1], "upper": auc_ci[2]},
])
```

Typical result: accuracy around 0.97-0.98 with a CI roughly ±0.02-0.03 wide, ROC-AUC very close to
1.0 with a narrow CI — this small, clean dataset gives a strong model, so the main teaching point is
the *method* (percentile bootstrap), which generalizes to much noisier production metrics where a
2-3 point CI width is the difference between "ship it" and "page someone."

**Step 3 — Real-feature batch split**

```python
train_median = X_train["worst area"].median()
low_mask = X_test["worst area"] <= train_median
high_mask = ~low_mask
print(low_mask.sum(), high_mask.sum())
```

`worst area` is strongly associated with malignancy (larger, more irregular nuclei tend to be
malignant), so splitting on it also implicitly skews the class balance and the distribution of
every correlated feature (`worst perimeter`, `mean area`, `worst concavity`, ...) between the two
batches — that's real, naturally occurring drift, not an injected synthetic shift.

**Step 4 — Per-feature KS tests with Bonferroni correction**

```python
n_features = len(X.columns)
alpha_corrected = 0.05 / n_features

rows = []
for col in X.columns:
    for batch_name, mask in [("low", low_mask), ("high", high_mask)]:
        stat, p = stats.ks_2samp(X_train[col], X_test.loc[mask, col])
        rows.append({"feature": col, "batch": batch_name, "statistic": stat, "p_value": p,
                     "significant_bonferroni": p < alpha_corrected})

ks_results = pd.DataFrame(rows)
ks_results.groupby("batch")["significant_bonferroni"].sum()
```

With `random_state=42` this comes back with the `high` batch (large-tumor-feature half) flagged on
more features (15/30) than `low` (13/30) — both batches show some drift since splitting on any
single feature reshuffles correlated features too, but `high` is the one enriched for the more
extreme, malignant-leaning tail.

**Step 5 — Per-batch metric CIs and verdict**

```python
def metrics_for_mask(mask):
    yt = y_test[mask].values
    yp = pd.Series(y_pred, index=y_test.index)[mask].values
    ypr = pd.Series(y_proba, index=y_test.index)[mask].values
    return {
        "accuracy": bootstrap_ci(yt, yp, accuracy_score),
        "f1": bootstrap_ci(yt, yp, f1_score),
        "roc_auc": bootstrap_ci(yt, ypr, roc_auc_score),
    }

low_metrics = metrics_for_mask(low_mask)
high_metrics = metrics_for_mask(high_mask)
```

**Verdict (example):** Both batches show some drift (splitting on `worst area` reshuffles correlated
features too), but `high` is flagged on more features at the Bonferroni-corrected threshold, and —
more importantly — it's the batch where the bootstrap CI for F1 actually dips (down to roughly 0.90
at the lower bound) while accuracy and ROC-AUC CIs also widen. The `low` batch, despite also being
"new" data, scores a perfect 1.0 on every metric with a degenerate (zero-width) CI, so it poses no
real quality risk. The statistical drift signal and the evaluation-metric uncertainty agree on
`high`, so that's the batch worth a page.

The general lesson: a KS-test drift alert by itself just says "the input distribution changed" — it
doesn't say whether that change hurts the model. Pairing it with a bootstrapped CI on the actual
evaluation metric, computed on the same slice, is what turns a statistical curiosity into an
actionable quality signal.

</details>
